In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(497128, 31)


In [2]:
product_features = (
    df.groupby("skuNumber")
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        avg_qty=(
            "effective_qty",
            "mean"
        )
    )
    .reset_index()
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty
0,SKU00001,1772,1594,558,1.111669
1,SKU00009,3294,2890,839,1.139792
2,SKU00010,13090,10232,3095,1.279320
3,SKU00020,1969,1634,742,1.205018
4,SKU00022,8286,7956,2710,1.041478


In [3]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category",
            "subCategory"
        ]
    ]
    .drop_duplicates()
)

product_features = (
    product_features.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory
0,SKU00001,1772,1594,558,1.111669,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum
1,SKU00001,1772,1594,558,1.111669,Chingles Filz Jar + 1 Rs.5 Lollipop,DS Group,Confectionery,Chewing Gum
2,SKU00009,3294,2890,839,1.139792,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum
3,SKU00009,3294,2890,839,1.139792,Chingles Maxi TF Jar + 1 Rs 5 Lollipop,DS Group,Confectionery,Chewing Gum
4,SKU00010,13090,10232,3095,1.279320,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix


In [4]:
total_retailers = (
    df["customerId"]
    .nunique()
)

product_features[
    "popularity_score"
] = (
    product_features[
        "unique_buyers"
    ]
    / total_retailers
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory,popularity_score
0,SKU00001,1772,1594,558,1.111669,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum,0.064583
1,SKU00001,1772,1594,558,1.111669,Chingles Filz Jar + 1 Rs.5 Lollipop,DS Group,Confectionery,Chewing Gum,0.064583
2,SKU00009,3294,2890,839,1.139792,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum,0.097106
3,SKU00009,3294,2890,839,1.139792,Chingles Maxi TF Jar + 1 Rs 5 Lollipop,DS Group,Confectionery,Chewing Gum,0.097106
4,SKU00010,13090,10232,3095,1.279320,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.358218


In [5]:
(
    product_features[
        [
            "itemName",
            "popularity_score"
        ]
    ]
    .sort_values(
        "popularity_score",
        ascending=False
    )
    .head(10)
)

,itemName,popularity_score
15,Rajnigandha 4g,0.652315
14,Rajnigandha 4g,0.652315
26,Tulsi 00 (with Silver) 0.45g,0.641435
25,TZ 00 (with Silver) 0.45g,0.641435
17,RG Pearl Elaichi MP Pouch 0.14g,0.433681
23,TZ 00 4.25g (6 Pcs),0.378472
24,Tulsi 00 4.25g (6 Pcs),0.378472
257,Rajnigandha 17g 2 Pcs,0.371528
258,Rajnigandha 17g 2 Pcs,0.371528
4,Pass Pass Meetha Magic 11.75g Hanger,0.358218


In [6]:
product_features[
    "category_rank"
] = (
    product_features
    .groupby("category")
    ["total_qty"]
    .rank(
        ascending=False,
        method="dense"
    )
)

product_features.head()

,skuNumber,total_qty,total_orders,unique_buyers,avg_qty,itemName,brand,category,subCategory,popularity_score,category_rank
0,SKU00001,1772,1594,558,1.111669,Chingles Filz Jar,DS Group,Confectionery,Chewing Gum,0.064583,13.0
1,SKU00001,1772,1594,558,1.111669,Chingles Filz Jar + 1 Rs.5 Lollipop,DS Group,Confectionery,Chewing Gum,0.064583,13.0
2,SKU00009,3294,2890,839,1.139792,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,Chewing Gum,0.097106,7.0
3,SKU00009,3294,2890,839,1.139792,Chingles Maxi TF Jar + 1 Rs 5 Lollipop,DS Group,Confectionery,Chewing Gum,0.097106,7.0
4,SKU00010,13090,10232,3095,1.279320,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,Mouth Freshener Mix,0.358218,3.0


In [7]:
product_features.to_parquet(
    "../data/processed/product_features.parquet",
    index=False
)

print("Saved")

Saved
